# Anthropic Claude

**Anthropic** builds the **Claude** model family with emphasis on safety, helpfulness, and long-context document work. Claude is widely used for coding assistants, enterprise chat, legal and financial document analysis, and research workflows.

Claude's **Messages API** has distinct conventions — especially the separate `system` parameter and strict user/assistant alternation. This notebook covers Claude's models, API design, and strengths relative to OpenAI and Gemini.


---

## 1. Anthropic's Approach

Anthropic researches **AI safety** and **Constitutional AI** — training methods aimed at producing models that are helpful, honest, and harmless. For developers, the practical impact is:

- Strong performance on **long documents** and nuanced instructions
- Reliable **coding** and analysis tasks
- Clear, well-documented **Messages API**
- Enterprise adoption through direct API and cloud marketplaces (e.g. AWS Bedrock)

Claude competes directly with GPT-4o and Gemini Pro on general tasks while often leading on long-context and careful reasoning workloads.


---

## 2. Model Families

| Model tier | Typical use | Notes |
|------------|-------------|-------|
| **Claude Opus** | Highest capability | Complex analysis, hard reasoning |
| **Claude Sonnet** | Balanced default | Best speed/quality tradeoff for most apps |
| **Claude Haiku** | Fast, low-cost | High-volume, simpler tasks |

Model IDs include version dates (e.g. `claude-sonnet-4-20250514`). Check [Anthropic docs](https://docs.anthropic.com/en/docs/about-claude/models) for current names.

### When to choose Claude

| Use case | Why Claude |
|----------|------------|
| Long PDF / contract analysis | Large context windows |
| Careful writing and editing | Strong instruction following |
| Code review and refactoring | Competitive coding performance |
| Enterprise with Bedrock | Available on AWS with IAM auth |


---

## 3. Messages API Design

| Item | Value |
|------|-------|
| **Base URL** | `https://api.anthropic.com` |
| **Auth** | `x-api-key` header (`ANTHROPIC_API_KEY`) |
| **Endpoint** | `POST /v1/messages` |
| **Python SDK** | `pip install anthropic` |

```python
import anthropic

client = anthropic.Anthropic()  # ANTHROPIC_API_KEY from env

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    temperature=0.3,
    system="You are a concise technical writer.",
    messages=[
        {"role": "user", "content": "What is an API in two sentences?"},
    ],
)

for block in message.content:
    if block.type == "text":
        print(block.text)
```

### 3.1 System prompt is separate

Unlike OpenAI, Claude's **system** instruction is a top-level parameter — not a message in the list:

```python
system="You are a coding tutor."
messages=[{"role": "user", "content": "Explain list comprehensions."}]
```

### 3.2 User/assistant alternation

The `messages` list must alternate **user** and **assistant** turns. You include prior assistant replies when continuing a conversation.


---

## 4. Response Structure

Claude returns `content` as a **list of blocks**, not a single string:

| Block type | Meaning |
|------------|---------|
| `text` | Generated text (most common) |
| `tool_use` | Model requests a tool invocation |
| `thinking` | Extended thinking (model-dependent) |

Extract text in application code:

```python
text = " ".join(b.text for b in message.content if b.type == "text")
```

**Usage:** `message.usage.input_tokens` and `message.usage.output_tokens`.

**Stop reason:** `message.stop_reason` (e.g. `end_turn`, `max_tokens`).


---

## 5. Key Features

### 5.1 Tool use

Claude supports **tool use** (function calling) — the model emits structured `tool_use` blocks; your app runs the tool and returns results in a follow-up message.

### 5.2 Streaming

```python
with client.messages.stream(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[{"role": "user", "content": "Count to five."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
```

### 5.3 Long context

Claude models support very large context windows (model-dependent), making them suitable for full document ingestion without aggressive chunking — though RAG remains valuable for accuracy and cost.


---

## 6. Claude vs OpenAI vs Gemini

| Aspect | Claude | OpenAI | Gemini |
|--------|--------|--------|--------|
| System prompt | `system=` param | `role: system` message | `system_instruction` |
| Response shape | Content blocks | Single `message.content` | `response.text` |
| Auth header | `x-api-key` | `Bearer` | API key in client |
| Max output param | `max_tokens` | `max_tokens` | `max_output_tokens` |


---

## 7. Hands-On in This Course

| Path | Demonstrates |
|------|--------------|
| `../04. Anthropic Claude/01. Console Application/` | Direct SDK: messages, streaming, token count |
| `../04. Anthropic Claude/02. Fast API Application/` | FastAPI: HTTP API, retries, SSE |

**Setup:**

1. Account at [console.anthropic.com](https://console.anthropic.com)
2. Set `ANTHROPIC_API_KEY` in repo root `.env`

```bash
python "../04. Anthropic Claude/01. Console Application/01_chat_completion.py"
cd "../04. Anthropic Claude/02. Fast API Application"
uvicorn app.main:app --reload --port 8012
```


---

## 8. Summary

Claude offers a clean **Messages API** with separate system instructions and block-based responses. It excels at **long documents**, **coding**, and **careful instruction following**. Remember user/assistant alternation and extract text from content blocks in production code.
